In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

In [ ]:
DRIVE_DATASET_DIR = Path(
    "/content/drive/MyDrive/Cellula/EyeDisease/Data/dataset_splits_v2"
)

LOCAL_DATASET_DIR = Path(
    "/content/dataset_splits_v2"
)

TRAIN_DIR = LOCAL_DATASET_DIR / "train"
VAL_DIR   = LOCAL_DATASET_DIR / "val"
TEST_DIR  = LOCAL_DATASET_DIR / "test"


if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset not found on Google Drive:\n{DRIVE_DATASET_DIR}"
    )

if not LOCAL_DATASET_DIR.exists():
    print("Copying dataset from Google Drive to local /content...")

    shutil.copytree(
        DRIVE_DATASET_DIR,
        LOCAL_DATASET_DIR
    )

    print("✅ Dataset copied successfully.")
else:
    print("✅ Local dataset already exists. Skipping copy.")

Copying dataset from Google Drive to local /content...
✅ Dataset copied successfully.


In [ ]:
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Train directory not found: {TRAIN_DIR}")

if not VAL_DIR.exists():
    raise FileNotFoundError(f"Validation directory not found: {VAL_DIR}")

if not TEST_DIR.exists():
    raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")


def count_images(directory):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    return sum(
        1
        for file in directory.rglob("*")
        if file.is_file() and file.suffix.lower() in extensions
    )


train_count = count_images(TRAIN_DIR)
val_count   = count_images(VAL_DIR)
test_count  = count_images(TEST_DIR)


print("\n================ DATASET CHECK ================")
print(f"Train path      : {TRAIN_DIR}")
print(f"Validation path : {VAL_DIR}")
print(f"Test path       : {TEST_DIR}")

print(f"\nTrain images      : {train_count}")
print(f"Validation images : {val_count}")
print(f"Test images       : {test_count}")


================ DATASET CHECK ================
Train path      : /content/dataset_splits_v2/train
Validation path : /content/dataset_splits_v2/val
Test path       : /content/dataset_splits_v2/test

Train images      : 12300
Validation images : 1776
Test images       : 1776


In [ ]:
SEED = 42
BATCH_SIZE = 16
IMG_SIZE = (224, 224)
NUM_CLASSES = 8
EPOCHS = 20

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 12300 files belonging to 8 classes.
Found 1776 files belonging to 8 classes.
Found 1776 files belonging to 8 classes.


In [ ]:
class_names = train_ds.class_names

print("\nClasses:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

print("\nDataset sizes:")
print(f"Train batches: {len(train_ds)}")
print(f"Val batches  : {len(val_ds)}")
print(f"Test batches : {len(test_ds)}")


Classes:
0: AMD
1: Cataract
2: Diabetic Retinopathy
3: Glaucoma
4: Hypertension
5: Myopia
6: Normal
7: Others

Dataset sizes:
Train batches: 769
Val batches  : 111
Test batches : 111


In [ ]:
normalization_layer = layers.Rescaling(1.0 / 255.0)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds = test_ds.map(
    lambda x, y: (normalization_layer(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
original_train_counts = {
    "Normal": 3288,
    "Diabetic Retinopathy": 2879,
    "Others": 771,
    "Glaucoma": 651,
    "Cataract": 238,
    "Myopia": 206,
    "AMD": 192,
    "Hypertension": 62
}

final_balanced_counts = {
    "Normal": 3000,
    "Diabetic Retinopathy": 2800,
    "Others": 1500,
    "Glaucoma": 1500,
    "Cataract": 1000,
    "Myopia": 1000,
    "AMD": 1000,
    "Hypertension": 500
}


total_original_samples = sum(
    original_train_counts.values()
)

num_classes = len(class_names)


raw_weights = {}

for class_name, count in original_train_counts.items():

    raw_weight = np.sqrt(
        total_original_samples /
        (num_classes * count)
    )

    raw_weights[class_name] = raw_weight

In [ ]:
mean_raw_weight = np.mean(
    list(raw_weights.values())
)

normalized_weights = {
    class_name: weight / mean_raw_weight
    for class_name, weight in raw_weights.items()
}


soft_weights = {
    class_name: float(
        np.clip(weight, 0.5, 3.0)
    )
    for class_name, weight in normalized_weights.items()
}


class_weights = {
    class_names.index(class_name): weight
    for class_name, weight in soft_weights.items()
}


In [ ]:
print("\n================ CLASS WEIGHTS ================")

for class_index, weight in class_weights.items():

    print(
        f"{class_index}: "
        f"{class_names[class_index]:25s} -> "
        f"{weight:.4f}"
    )


================ CLASS WEIGHTS ================
6: Normal                    -> 0.5000
2: Diabetic Retinopathy      -> 0.5000
7: Others                    -> 0.6475
3: Glaucoma                  -> 0.7047
1: Cataract                  -> 1.1655
5: Myopia                    -> 1.2527
0: AMD                       -> 1.2976
4: Hypertension              -> 2.2834


In [ ]:
class_weight_df = pd.DataFrame({
    "class": class_names,
    "original_train_count": [
        original_train_counts[class_name]
        for class_name in class_names
    ],
    "final_balanced_train_count": [
        final_balanced_counts[class_name]
        for class_name in class_names
    ],
    "class_weight": [
        class_weights[index]
        for index in range(len(class_names))
    ]
})


class_weight_df

,class,original_train_count,final_balanced_train_count,class_weight
0,AMD,192,1000,1.297573
1,Cataract,238,1000,1.165450
2,Diabetic Retinopathy,2879,2800,0.500000
3,Glaucoma,651,1500,0.704680
4,Hypertension,62,500,2.283423
5,Myopia,206,1000,1.252705
6,Normal,3288,3000,0.500000
7,Others,771,1500,0.647523


In [ ]:
# =========================================================
# 15. Custom CNN
# =========================================================

def build_custom_cnn(
    input_shape=(224, 224, 3),
    num_classes=8
):

    model = models.Sequential([

        # Block 1
        layers.Conv2D(
            32,
            (3, 3),
            padding="same",
            activation="relu",
            input_shape=input_shape
        ),
        layers.BatchNormalization(),

        layers.Conv2D(
            32,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.MaxPooling2D((2, 2)),


        # Block 2
        layers.Conv2D(
            64,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.Conv2D(
            64,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.MaxPooling2D((2, 2)),


        # Block 3
        layers.Conv2D(
            128,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.Conv2D(
            128,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.MaxPooling2D((2, 2)),


        # Block 4
        layers.Conv2D(
            256,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.Conv2D(
            256,
            (3, 3),
            padding="same",
            activation="relu"
        ),
        layers.BatchNormalization(),

        layers.MaxPooling2D((2, 2)),


        # Classifier
        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.5),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.Dropout(0.3),

        layers.Dense(
            num_classes,
            activation="softmax"
        )
    ])

    return model


model = build_custom_cnn(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             

 Total params: 1,210,024 (4.62 MB)

 Trainable params: 1,208,104 (4.61 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [ ]:
# =========================================================
# 16. Compile Custom CNN
# =========================================================

LR = 1e-3

model.compile(
    optimizer=optimizers.Adam(
        learning_rate=LR
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================================================
# 17. CNN Callbacks
# =========================================================

CNN_SAVE_DIR = Path(
    "/content/drive/MyDrive/Cellula/EyeDisease/models/cnn"
)

CNN_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CNN_BEST_MODEL_PATH = (
    CNN_SAVE_DIR / "custom_cnn_best.keras"
)


cnn_checkpoint = callbacks.ModelCheckpoint(
    CNN_BEST_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

cnn_early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

cnn_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)


# =========================================================
# 18. Train Custom CNN
# =========================================================

history_cnn = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,

    # IMPORTANT
    class_weight=class_weights,

    callbacks=[
        cnn_checkpoint,
        cnn_early_stop,
        cnn_reduce_lr
    ]
)

Epoch 1/20
769/769 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.2449 - loss: 1.6356
Epoch 1: val_loss improved from None to 1.80354, saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_best.keras
769/769 ━━━━━━━━━━━━━━━━━━━━ 363s 164ms/step - accuracy: 0.2695 - loss: 1.5240 - val_accuracy: 0.2748 - val_loss: 1.8035 - learning_rate: 0.0010
Epoch 2/20
769/769 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.3143 - loss: 1.3959
Epoch 2: val_loss improved from 1.80354 to 1.54019, saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_best.keras
769/769 ━━━━━━━━━━━━━━━━━━━━ 312s 129ms/step - accuracy: 0.3277 - loss: 1.3768 - val_accuracy: 0.3953 - val_loss: 1.5402 - learning_rate: 0.0010
Epoch 3/20
769/769 ━━━━━━━━━━━━━

In [ ]:
# =========================================================
# 19. Save CNN Model + History
# =========================================================

CNN_FINAL_MODEL_PATH = (
    CNN_SAVE_DIR / "custom_cnn_final.keras"
)

model.save(CNN_FINAL_MODEL_PATH)


def history_to_dict(history):

    return {
        key: [float(x) for x in values]
        for key, values in history.history.items()
    }


cnn_history_data = history_to_dict(history_cnn)


CNN_HISTORY_PATH = (
    CNN_SAVE_DIR / "custom_cnn_training_history.json"
)

with open(
    CNN_HISTORY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        cnn_history_data,
        f,
        indent=4
    )


print("✅ CNN model saved to:")
print(CNN_FINAL_MODEL_PATH)

print("\n✅ CNN history saved to:")
print(CNN_HISTORY_PATH)

✅ CNN model saved to:
/content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_final.keras

✅ CNN history saved to:
/content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_training_history.json


In [ ]:
# تحميل الموديل
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_final.keras'
)

# تحميل ملف الـ history
with open('/content/drive/MyDrive/Cellula/EyeDisease/models/cnn/custom_cnn_training_history.json', 'r') as f:
    history = json.load(f)

# عرض المفاتيح الموجودة (accuracy, loss, val_accuracy, val_loss)
print("Keys:", history.keys())

Keys: dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])


In [ ]:
def evaluate_model_once(
    model,
    test_ds,
    class_names,
    model_name="Model"
):

    print(f"\n{'=' * 60}")
    print(f"{model_name} - TEST EVALUATION")
    print(f"{'=' * 60}")


    # -----------------------------------------------------
    # Generate predictions ONCE
    # -----------------------------------------------------

    y_pred_probs = model.predict(
        test_ds,
        verbose=1
    )

    y_pred = np.argmax(
        y_pred_probs,
        axis=1
    )


    # -----------------------------------------------------
    # Collect true labels ONCE
    # -----------------------------------------------------

    y_true = np.concatenate([
        labels.numpy()
        for _, labels in test_ds
    ])


    # -----------------------------------------------------
    # Accuracy
    # -----------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    # -----------------------------------------------------
    # Test Loss
    # -----------------------------------------------------

    test_loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true,
        y_pred_probs
    )

    test_loss = float(
        tf.reduce_mean(test_loss).numpy()
    )


    # -----------------------------------------------------
    # Precision / Recall / F1
    # -----------------------------------------------------

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )


    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )


    # -----------------------------------------------------
    # Classification Report
    # -----------------------------------------------------

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )


    print("\nClassification Report:\n")
    print(report)


    print(f"Test Loss       : {test_loss:.4f}")
    print(f"Test Accuracy   : {accuracy:.4f}")
    print(f"Macro Precision : {macro_precision:.4f}")
    print(f"Macro Recall    : {macro_recall:.4f}")
    print(f"Macro F1        : {macro_f1:.4f}")
    print(f"Weighted F1     : {weighted_f1:.4f}")


    # -----------------------------------------------------
    # Confusion Matrix
    # -----------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred
    )


    plt.figure(figsize=(10, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{model_name} - Confusion Matrix")

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.show()


    # -----------------------------------------------------
    # Return results
    # -----------------------------------------------------

    results = {
        "model": model_name,
        "loss": test_loss,
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1)
    }


    return results, y_true, y_pred, y_pred_probs

In [ ]:
# =========================================================
# 21. Evaluate Custom CNN
# =========================================================

cnn_results, cnn_y_true, cnn_y_pred, cnn_y_pred_probs = (
    evaluate_model_once(
        model=model,
        test_ds=test_ds,
        class_names=class_names,
        model_name="Custom CNN"
    )
)


Custom CNN - TEST EVALUATION
111/111 ━━━━━━━━━━━━━━━━━━━━ 285s 3s/step


In [ ]:
# =========================================================
# 22. Optional Memory Cleanup
# =========================================================

del train_ds
del val_ds

gc.collect()

tf.keras.backend.clear_session()

print("✅ Training datasets cleared from memory.")

In [ ]:
# =========================================================
# 23. ResNet50 Dataset Loading
# =========================================================

train_ds_rn = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED
)

val_ds_rn = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds_rn = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)


# =========================================================
# ResNet50 preprocessing
# =========================================================

train_ds_rn = train_ds_rn.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds_rn = val_ds_rn.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_ds_rn = test_ds_rn.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)


# =========================================================
# Performance tuning
# NO CACHE
# =========================================================

train_ds_rn = (
    train_ds_rn
    .shuffle(1000)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_rn = (
    val_ds_rn
    .prefetch(tf.data.AUTOTUNE)
)

test_ds_rn = (
    test_ds_rn
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# =========================================================
# 24. Build ResNet50
# =========================================================

def build_resnet50(
    input_shape=(224, 224, 3),
    num_classes=8
):

    base_model = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )

    # Freeze pretrained backbone
    base_model.trainable = False

    inputs = layers.Input(
        shape=input_shape
    )

    x = base_model(
        inputs,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.BatchNormalization()(x)

    x = layers.Dense(
        256,
        activation="relu"
    )(x)

    x = layers.Dropout(0.5)(x)

    x = layers.Dense(
        64,
        activation="relu"
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = models.Model(
        inputs,
        outputs,
        name="resnet50_eye"
    )

    return model


resnet_model = build_resnet50(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

resnet_model.summary()

In [ ]:
# =========================================================
# 25. ResNet Stage 1
# =========================================================

resnet_model.compile(
    optimizer=optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================================================
# Save Directory
# =========================================================

SAVE_DIR = Path(
    "/content/drive/MyDrive/Cellula/EyeDisease/models/resnet"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    SAVE_DIR / "resnet50_best.keras"
)


# =========================================================
# Callbacks
# =========================================================

checkpoint = callbacks.ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)


# =========================================================
# Stage 1 Training
# =========================================================

history_stage1 = resnet_model.fit(
    train_ds_rn,
    validation_data=val_ds_rn,
    epochs=10,

    # IMPORTANT
    class_weight=class_weights,

    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

In [ ]:
# =========================================================
# 26. ResNet Stage 2 - Fine Tuning
# =========================================================

base_model = resnet_model.get_layer(
    "resnet50"
)

base_model.trainable = True


# Freeze all except last 30 layers

for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True


# =========================================================
# Recompile
# =========================================================

resnet_model.compile(
    optimizer=optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================================================
# Fine-tuning Callbacks
# =========================================================

fine_tune_checkpoint = callbacks.ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

fine_tune_early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

fine_tune_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)


# =========================================================
# Stage 2 Training
# =========================================================

history_stage2 = resnet_model.fit(
    train_ds_rn,
    validation_data=val_ds_rn,
    epochs=10,

    # IMPORTANT
    class_weight=class_weights,

    callbacks=[
        fine_tune_checkpoint,
        fine_tune_early_stop,
        fine_tune_reduce_lr
    ]
)

In [ ]:
# =========================================================
# 27. Save Final ResNet Model
# =========================================================

FINAL_MODEL_PATH = (
    SAVE_DIR / "resnet50_eye_final.keras"
)

resnet_model.save(
    FINAL_MODEL_PATH
)

print("✅ Final ResNet model saved to:")
print(FINAL_MODEL_PATH)


# =========================================================
# 28. Save Training Histories
# =========================================================

def history_to_dict(history):

    return {
        key: [float(x) for x in values]
        for key, values in history.history.items()
    }


history_data = {
    "stage1": history_to_dict(
        history_stage1
    ),

    "stage2": history_to_dict(
        history_stage2
    )
}


HISTORY_PATH = (
    SAVE_DIR /
    "resnet50_training_history.json"
)


with open(
    HISTORY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history_data,
        f,
        indent=4
    )


print("✅ ResNet history saved to:")
print(HISTORY_PATH)

In [ ]:
# =========================================================
# 29. Memory Cleanup Before Final Evaluation
# =========================================================

del train_ds_rn
del val_ds_rn

gc.collect()

print("✅ Training and validation datasets removed.")

In [ ]:
fine_tune_checkpoint = callbacks.ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

fine_tune_early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

fine_tune_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

In [ ]:
history_stage2 = resnet_model.fit(
    train_ds_rn,
    validation_data=val_ds_rn,
    epochs=10,
    callbacks=[
        fine_tune_checkpoint,
        fine_tune_early_stop,
        fine_tune_reduce_lr
    ]
)

Epoch 1/10
1538/1538 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.5658 - loss: 1.1414
Epoch 1: val_loss improved from None to 0.89528, saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_best.keras
1538/1538 ━━━━━━━━━━━━━━━━━━━━ 317s 192ms/step - accuracy: 0.5817 - loss: 1.1035 - val_accuracy: 0.6385 - val_loss: 0.8953 - learning_rate: 1.0000e-05
Epoch 2/10
1537/1538 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.6205 - loss: 0.9844
Epoch 2: val_loss improved from 0.89528 to 0.87209, saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_best.keras
1538/1538 ━━━━━━━━━━━━━━━━━━━━ 295s 192ms/step - accuracy: 0.6260 - loss: 0.9705 - val_accuracy: 0.6453 - val_loss: 0.8721 - learning_rate: 1.0000e-05
Epoch 3/10


In [ ]:
# =========================================================
# Save final model
# =========================================================

FINAL_MODEL_PATH = SAVE_DIR / "resnet50_eye_final.keras"

resnet_model.save(FINAL_MODEL_PATH)

print(f"✅ Final model saved to:")
print(FINAL_MODEL_PATH)

✅ Final model saved to:
/content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_eye_final.keras


In [ ]:
# =========================================================
# Save histories
# =========================================================

def history_to_dict(history):
    return {
        key: [float(x) for x in values]
        for key, values in history.history.items()
    }


history_data = {
    "stage1": history_to_dict(history_stage1),
    "stage2": history_to_dict(history_stage2)
}

HISTORY_PATH = SAVE_DIR / "resnet50_training_history.json"

with open(HISTORY_PATH, "w") as f:
    json.dump(
        history_data,
        f,
        indent=4
    )

print(f"✅ History saved to:")
print(HISTORY_PATH)

✅ History saved to:
/content/drive/MyDrive/Cellula/EyeDisease/models/resnet/resnet50_training_history.json
